# AI Capability Progression — Calibrated Sigmoid Model
**Author:** Dragos | **Last updated:** April 2026  
**Purpose:** Fit a data-driven sigmoid curve to real published AI benchmark scores  
and project future AI capability with quantified uncertainty.  
**Key data point:** Claude Mythos Preview — SWE-bench Verified 93.9% (April 7, 2026)

> *All benchmark data sourced from published model cards and Papers with Code.*  
> *No hardcoded parameters — every parameter is estimated from data.*


In [ ]:
# ── Environment setup
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # add project root to path

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook_connected"

from src.data.ai_benchmarks import get_benchmark_dataframe, BENCHMARK_META, get_swe_bench_series
from src.models.benchmark_curve import BenchmarkCurveFitter
from src.viz.plots import plot_benchmark_progression

print("✓ Imports OK")


## 1. Load Historical Benchmark Data
All data points are from published sources with citations in `src/data/ai_benchmarks.py`.


In [ ]:
df = get_benchmark_dataframe(normalize=True)
print(f"Total benchmark entries: {len(df)}")
print(f"Benchmarks: {df['benchmark'].unique()}")
print()
print("SWE-bench data (sorted by year):")
df[df['benchmark']=='swe_bench'][['year','model','organization','score','source']].to_string(index=False)


## 2. Fit Sigmoid Curves to Each Benchmark
Using `scipy.optimize.curve_fit` with:
- **95% confidence intervals** via t-distribution
- **Monte Carlo uncertainty propagation** through parameter covariance matrix
- R² and RMSE as goodness-of-fit metrics


In [ ]:
fitter = BenchmarkCurveFitter(model="sigmoid")
fit_results = fitter.fit_all_benchmarks(df)

print("═" * 65)
for bm, res in fit_results.items():
    label = BENCHMARK_META[bm]['label']
    print(f"\n📊 {label}")
    print(f"   R² = {res.r_squared:.4f}  |  RMSE = {res.rmse:.4f}  |  n = {res.n_points}")
    print(f"   Inflection year: {res.inflection_year():.1f}")
    y99 = res.year_to_reach(0.99)
    print(f"   Year to reach 99%: {y99:.1f}" if y99 else "   Year to reach 99%: beyond projection window")
    print()
    print(res.summary().to_string(index=False))
    print("─" * 65)


## 3. SWE-bench Deep Dive
SWE-bench Verified is the most labor-market-relevant benchmark:  
it measures *autonomous resolution of real software engineering tasks*.  
A score of 93.9% (Mythos, April 2026) means the model can resolve ~94% of  
real GitHub issues without human intervention.


In [ ]:
# Plot SWE-bench with fitted curve and uncertainty band
proj_years = np.linspace(2021, 2035, 500)
fig = plot_benchmark_progression(
    df,
    fit_results=fit_results,
    projection_years=proj_years,
    benchmark="swe_bench",
)
fig.update_layout(height=520)
fig.show()


## 4. Multi-Benchmark Comparison


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

benchmarks = ["swe_bench", "humaneval", "mmlu"]
labels = ["SWE-bench Verified", "HumanEval", "MMLU"]
colors = ["#58a6ff", "#3fb950", "#f78166"]
proj = np.linspace(2020, 2035, 400)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=labels,
    shared_yaxes=True,
)

for col_idx, (bm, color) in enumerate(zip(benchmarks, colors), start=1):
    bm_df = df[df["benchmark"] == bm]
    mythos = bm_df[bm_df["model"] == "Claude Mythos Preview"]
    normal = bm_df[bm_df["model"] != "Claude Mythos Preview"]

    if bm in fit_results:
        res = fit_results[bm]
        mean, lo, hi = res.predict_with_uncertainty(proj)
        fig.add_trace(go.Scatter(
            x=np.concatenate([proj, proj[::-1]]),
            y=np.concatenate([hi * 100, (lo * 100)[::-1]]),
            fill="toself", fillcolor=f"rgba(88,166,255,0.10)",
            line=dict(width=0), showlegend=False, hoverinfo="skip",
        ), row=1, col=col_idx)
        fig.add_trace(go.Scatter(
            x=proj, y=mean * 100,
            line=dict(color=color, width=2, dash="dash"),
            name=f"{labels[col_idx-1]} fit", showlegend=False,
        ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=normal["year"], y=normal["score"],
        mode="markers", marker=dict(size=8, color=color, opacity=0.85),
        name=labels[col_idx-1], showlegend=(col_idx==1),
    ), row=1, col=col_idx)

    if not mythos.empty:
        fig.add_trace(go.Scatter(
            x=mythos["year"], y=mythos["score"],
            mode="markers", marker=dict(size=14, color="#d2a8ff", symbol="star"),
            name="Mythos ⭐" if col_idx == 1 else "Mythos ⭐",
            showlegend=(col_idx == 1),
        ), row=1, col=col_idx)

fig.update_layout(
    height=450, title="<b>AI Capability Progression Across Benchmarks</b>",
    paper_bgcolor="#161b22", plot_bgcolor="#0d1117",
    font=dict(color="#e6edf3"),
)
fig.update_yaxes(title_text="Score (%)", range=[0, 105], row=1, col=1)
fig.show()


## 5. Key Findings

| Benchmark | R² | Inflection Year | Year to 99% |
|-----------|-----|----------------|-------------|
| SWE-bench Verified | — | — | — |
| HumanEval | — | — | — |
| MMLU | — | — | — |

> *Run the cells above to populate this table.*

**Interpretation:**
- The sigmoid inflection point represents the **year of fastest capability growth**.
- Mythos Preview (April 2026) confirms the SWE-bench curve is in its **steepest phase**.
- Projection to 99% marks when AI becomes indistinguishable from human experts on these tasks.

**Next:** → `03_employment_impact.ipynb` uses these fitted curves to drive unemployment projections.
